In [1]:
import time
import numpy as np
import pandas as pd
import yaml
from sentence_transformers import SentenceTransformer

with open("../configs/project_config.yaml") as f:
    config = yaml.safe_load(f)

MODEL_NAME = config["embedding_model"]["name"]
EXPECTED_DIM = config["embedding_model"]["dimension"]

products = pd.read_parquet("../data/processed/products.parquet")
judgments = pd.read_parquet("../data/processed/judgments.parquet")

t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
print(f"loaded {MODEL_NAME} in {time.time()-t0:.1f}s")
print("device:", model.device)

products_small = products.sample(10_000, random_state=42).reset_index(drop=True)
queries_small = judgments.drop_duplicates("query_id").sample(100, random_state=42)["query"].tolist()

t0 = time.time()
product_emb_small = model.encode(products_small["product_text"].tolist(), batch_size=64, show_progress_bar=True)
print(f"encoded {len(products_small)} products in {time.time()-t0:.1f}s")

print("embedding shape:", product_emb_small.shape)
assert product_emb_small.shape[1] == EXPECTED_DIM, f"expected dim {EXPECTED_DIM}, got {product_emb_small.shape[1]}"

norms = np.linalg.norm(product_emb_small, axis=1)
print("norm mean/min/max (before normalization):", norms.mean(), norms.min(), norms.max())


KeyboardInterrupt: 

In [2]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("model device:", model.device)


torch version: 2.11.0+cu128
cuda available: True
model device: cuda:0


In [3]:
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_fp16 = model.half()

t0 = time.time()
product_emb_fp16 = model_fp16.encode(
    products_small["product_text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
)
elapsed = time.time() - t0
print(f"fp16, batch=128: encoded 10000 in {elapsed:.1f}s -> {elapsed/10000*1000:.1f}ms/product")

norms_fp16 = np.linalg.norm(product_emb_fp16, axis=1)
print("norm mean/min/max:", norms_fp16.mean(), norms_fp16.min(), norms_fp16.max())


NVIDIA GeForce GTX 1650
VRAM: 4.3 GB


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

fp16, batch=128: encoded 10000 in 1278.6s -> 127.9ms/product
norm mean/min/max: 1.0 0.9995 1.0


In [4]:
val_ids = set(pd.read_csv("../data/processed/splits/validation_query_ids.csv")["query_id"])
test_ids = set(pd.read_csv("../data/processed/splits/test_query_ids.csv")["query_id"])
eval_ids = val_ids | test_ids

required_product_ids = set(judgments.loc[judgments["query_id"].isin(eval_ids), "product_id"])
print("products required for val+test eval:", len(required_product_ids))

TARGET_SIZE = 100_000
remaining_budget = TARGET_SIZE - len(required_product_ids)
print("remaining budget for random fill:", remaining_budget)

if remaining_budget > 0:
    fill_pool = products.loc[~products["product_id"].isin(required_product_ids), "product_id"]
    fill_ids = fill_pool.sample(remaining_budget, random_state=config["project"]["seed"])
    keep_ids = required_product_ids | set(fill_ids)
else:
    keep_ids = required_product_ids
    print("required products alone exceed target; keeping all, no random fill")

products_100k = products[products["product_id"].isin(keep_ids)].reset_index(drop=True)
print("final catalogue size:", len(products_100k))

products_100k.to_parquet("../data/processed/products_100k.parquet", index=False)
print("saved data/processed/products_100k.parquet")


products required for val+test eval: 164006
remaining budget for random fill: -64006
required products alone exceed target; keeping all, no random fill
final catalogue size: 164006
saved data/processed/products_100k.parquet


In [5]:
import os
os.rename("../data/processed/products_100k.parquet", "../data/processed/products_164k.parquet")
print("renamed to products_164k.parquet")

model = model.float()  # undo the in-place .half() mutation
print("model dtype:", next(model.parameters()).dtype)  # must print torch.float32


renamed to products_164k.parquet
model dtype: torch.float32


In [6]:
products_164k = pd.read_parquet("../data/processed/products_164k.parquet")
print("catalogue size:", len(products_164k))

t0 = time.time()
product_embeddings = model.encode(
    products_164k["product_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
)
elapsed = time.time() - t0
print(f"encoded {len(products_164k)} products in {elapsed/60:.1f} min -> {elapsed/len(products_164k)*1000:.1f}ms/product")

norms = np.linalg.norm(product_embeddings, axis=1)
print("norm mean/min/max:", norms.mean(), norms.min(), norms.max())

os.makedirs("../artifacts/indexes", exist_ok=True)
np.save("../artifacts/indexes/product_embeddings.npy", product_embeddings.astype(np.float32))
products_164k[["product_id"]].to_csv("../artifacts/indexes/product_embedding_ids.csv", index=False)
print("saved embeddings + ID order file")


catalogue size: 164006


Batches:   0%|          | 0/2563 [00:00<?, ?it/s]

encoded 164006 products in 71.2 min -> 26.1ms/product
norm mean/min/max: 1.0 0.9999999 1.0000001
saved embeddings + ID order file


In [7]:
eval_query_df = (
    judgments[judgments["query_id"].isin(eval_ids)]
    .drop_duplicates("query_id")[["query_id", "query"]]
    .reset_index(drop=True)
)
print("queries to encode:", len(eval_query_df))

t0 = time.time()
query_embeddings = model.encode(eval_query_df["query"].tolist(), batch_size=64, show_progress_bar=True)
elapsed = time.time() - t0
print(f"encoded {len(eval_query_df)} queries in {elapsed:.1f}s")

norms_q = np.linalg.norm(query_embeddings, axis=1)
print("query norm mean/min/max:", norms_q.mean(), norms_q.min(), norms_q.max())

np.save("../artifacts/indexes/query_embeddings.npy", query_embeddings.astype(np.float32))
eval_query_df[["query_id"]].to_csv("../artifacts/indexes/query_embedding_ids.csv", index=False)
print("saved query embeddings + ID order file")


queries to encode: 8954


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

encoded 8954 queries in 12.5s
query norm mean/min/max: 1.0 0.9999999 1.0000001
saved query embeddings + ID order file


In [8]:
import json
from datetime import datetime, timezone

metadata = {
    "model_name": MODEL_NAME,
    "dimension": EXPECTED_DIM,
    "batch_size": 64,
    "device": str(model.device),
    "normalized": True,
    "catalogue_size": len(products_164k),
    "n_queries_encoded": len(eval_query_df),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
with open("../artifacts/embedding_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))


{
  "model_name": "BAAI/bge-base-en-v1.5",
  "dimension": 768,
  "batch_size": 64,
  "device": "cuda:0",
  "normalized": true,
  "catalogue_size": 164006,
  "n_queries_encoded": 8954,
  "created_at": "2026-08-18T20:04:42.333653+00:00"
}


In [9]:
test_query = "waterproof shoes"
test_emb = model.encode([test_query])[0]
scores = product_embeddings @ test_emb  # normalized vectors -> dot product == cosine similarity
top_idx = np.argsort(-scores)[:10]

print(f"Top 10 nearest products for '{test_query}':")
for idx in top_idx:
    print(f"  [{scores[idx]:.3f}] {products_164k.iloc[idx]['product_title']}")

assert len(product_embeddings) == len(products_164k)
saved_ids = pd.read_csv("../artifacts/indexes/product_embedding_ids.csv")
sample_idx = np.random.RandomState(42).randint(0, len(products_164k), 5)
for idx in sample_idx:
    assert saved_ids.iloc[idx]["product_id"] == products_164k.iloc[idx]["product_id"]
print("random ID-alignment check passed")


Top 10 nearest products for 'waterproof shoes':
  [0.754] ARUNNERS Women Rain Boots for Girls (Clear, 2XL)
  [0.750] Water Shoes Mens Womens Beach Quick Dry Swim Barefoot Shoes Aqua Sock Outdoor Athletic Pool Shoes for Kayaking, Swimming, Surfing, Yoga, Fishing(355Black/Grey-44EU)
  [0.746] Womens and Mens Kids Water Shoes Barefoot Quick-Dry Aqua Socks for Beach Swim Surf Yoga Exercise (Undersea Shark, M)
  [0.745] ARUNNERS Motorcycle Rain Gear for Men (Black, 3XL)
  [0.745] Water Shoes for Womens and Mens Summer Barefoot Shoes Quick Dry Aqua Socks for Beach Swim Yoga Exercise (Orange/Blue, 42/43)
  [0.745] HOBIBEAR Toddler Boys Girls Water Shoes Quick Dry Closed-Toe Aquatic Sport Sandals (Black-Orange,11 Little Kid)
  [0.744] Alibress Quick Dry Sport Water Shoes for Men Breathable Aqua Water Shoes Black 9.5 M US
  [0.744] Womens and Mens Kids Water Shoes Barefoot Quick-Dry Aqua Socks for Beach Swim Surf Yoga Exercise (Shark, L)
  [0.744] Water Shoes for Womens and Mens Summer Barefoot

In [2]:
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-base-en-v1.5"

full_products = pd.read_parquet("../data/processed/products.parquet")
print("full catalogue:", len(full_products))

existing_ids = pd.read_csv("../artifacts/indexes/product_embedding_ids.csv")["product_id"]
existing_embeddings = np.load("../artifacts/indexes/product_embeddings.npy")
existing_lookup = dict(zip(existing_ids, existing_embeddings))
print("already embedded:", len(existing_lookup))

new_products = full_products[~full_products["product_id"].isin(existing_lookup.keys())].reset_index(drop=True)
print("new products to encode:", len(new_products))

model = SentenceTransformer(MODEL_NAME)
print("device:", model.device)

t0 = time.time()
new_embeddings = model.encode(
    new_products["product_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
)
elapsed = time.time() - t0
print(f"encoded {len(new_products)} new products in {elapsed/60:.1f} min -> {elapsed/len(new_products)*1000:.1f}ms/product")

new_lookup = dict(zip(new_products["product_id"], new_embeddings))
combined_lookup = {**existing_lookup, **new_lookup}

final_embeddings = np.stack([combined_lookup[pid] for pid in full_products["product_id"]]).astype(np.float32)
print("final embeddings shape:", final_embeddings.shape)

norms = np.linalg.norm(final_embeddings, axis=1)
print("norm mean/min/max:", norms.mean(), norms.min(), norms.max())

np.save("../artifacts/indexes/product_embeddings.npy", final_embeddings)
full_products[["product_id"]].to_csv("../artifacts/indexes/product_embedding_ids.csv", index=False)
print("saved full-catalogue embeddings (overwrote the 164k version)")


full catalogue: 482105
already embedded: 164006
new products to encode: 318099


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

device: cuda:0


Batches:   0%|          | 0/4971 [00:00<?, ?it/s]

encoded 318099 new products in 134.3 min -> 25.3ms/product
final embeddings shape: (482105, 768)
norm mean/min/max: 1.0 0.9999999 1.0000001
saved full-catalogue embeddings (overwrote the 164k version)


In [3]:
all_query_ids = judgments.drop_duplicates("query_id")[["query_id", "query"]].reset_index(drop=True)
print("total unique queries:", len(all_query_ids))

t0 = time.time()
all_query_embeddings = model.encode(all_query_ids["query"].tolist(), batch_size=64, show_progress_bar=True)
elapsed = time.time() - t0
print(f"encoded {len(all_query_ids)} queries in {elapsed:.1f}s")

norms_q = np.linalg.norm(all_query_embeddings, axis=1)
print("norm mean/min/max:", norms_q.mean(), norms_q.min(), norms_q.max())

np.save("../artifacts/indexes/query_embeddings.npy", all_query_embeddings.astype(np.float32))
all_query_ids[["query_id"]].to_csv("../artifacts/indexes/query_embedding_ids.csv", index=False)
print("saved ALL query embeddings (train+val+test) - overwrote the val+test-only version")


total unique queries: 29844


Batches:   0%|          | 0/467 [00:00<?, ?it/s]

encoded 29844 queries in 38.9s
norm mean/min/max: 1.0 0.9999998 1.0000001
saved ALL query embeddings (train+val+test) - overwrote the val+test-only version
